In [ ]:
# 1. Atualizar o gerenciador e instalar o zstd
!apt-get update -qq
!apt-get install -y zstd -qq

# 2. Instalar pacotes Python necessários
!pip install pdfplumber ollama tqdm -q

# 3. Instalar o Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# 1. Iniciar o servidor Ollama em segundo plano
subprocess.Popen(["ollama", "serve"])

# 2. Aguardar 5 segundos para o servidor inicializar completamente
time.sleep(5)

# 3. Baixar o modelo (ajuste para o modelo que você deseja usar)
!ollama pull llama3.2:latest

In [ ]:
import json
import re
import pdfplumber
import ollama
from tqdm import tqdm

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

def split_text(text, chunk_size=1800):
    chunks = []
    cur = ""
    for p in text.splitlines():
        p = p.strip()
        if not p:
            continue
        if len(cur) + len(p) + 1 <= chunk_size:
            cur += p + "\n"
        else:
            if cur:
                chunks.append(cur.strip())
            cur = p + "\n"
    if cur:
        chunks.append(cur.strip())
    return chunks

def generate_instruction_response(
    chunk,
    model="gemma4:31b-cloud",
    questions_per_chunk=5,
    max_question_tokens=50,
    max_answer_tokens=50,
    temperature=1.0,
    top_p=0.95,
    repeat_penalty=1.15
):
    """
    Gera múltiplos pares (question, answer) por bloco,
    usando o prompt detalhado para garantir diversidade.
    """
    prompt = f"""
You are an expert dataset generator for training and fine-tuning large language models.

Your task is to generate EXACTLY {questions_per_chunk} high-quality Question-Answer pairs using ONLY the information below.

========================================================
GENERAL RULES (for EVERY pair)
========================================================

• Each question: maximum {max_question_tokens} tokens.
• Each answer: maximum {max_answer_tokens} tokens.
• Never exceed these limits.
• Use ONLY the provided content.
• Never invent information.
• Never hallucinate.
• Never copy complete sentences.
• Rewrite naturally.

========================================================
DIVERSITY (apply to EACH pair independently)
========================================================

For EACH question you generate, imagine it comes from a RANDOMLY selected user profile:

- beginner
- student
- engineer
- technician
- customer
- researcher
- teacher
- maintenance worker
- operator
- curious person
- expert

Then, for EACH question, RANDOMLY choose ONE question style:

- definition
- explanation
- comparison
- procedure
- recommendation
- troubleshooting
- purpose
- characteristics
- summary
- practical use
- limitation
- cause
- effect
- example
- identification
- sequence
- condition
- yes/no
- what happens if
- advantages
- disadvantages
- differences
- interpretation
- best practice
- usage

========================================================
QUESTION STYLE (language variation)
========================================================

Use natural English.
Prefer common vocabulary.
Avoid rare words.
Avoid repetitive sentence structures.

Do NOT always begin with "What is...".
Vary the opening among these (and others you find natural):

What...
Why...
How...
When...
Where...
Who...
Which...
Can...
Could...
Should...
Would...
Does...
Do...
Explain...
Describe...
Summarize...
Compare...
Identify...
List...
Give...
Name...
What happens if...
Why is...
How can...
How should...
Under what conditions...
In which situations...
What is the purpose...
What are the advantages...
What are the disadvantages...
How does ... differ...
Why would someone...
What should be done...
What occurs before...
What occurs after...

========================================================
OUTPUT FORMAT
========================================================

Repeat the following block EXACTLY {questions_per_chunk} times:

INSTRUCTION: <your question here>

RESPONSE: <your answer here>

========================================================
CONTENT (use only this)
========================================================

{chunk}
"""
    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "top_p": top_p,
            "repeat_penalty": repeat_penalty
        }
    )
    content = resp["message"]["content"]

    # Captura todos os blocos INSTRUCTION / RESPONSE
    matches = re.findall(
        r"INSTRUCTION:\s*(.*?)\s*RESPONSE:\s*(.*?)(?=INSTRUCTION:|$)",
        content,
        flags=re.S
    )
    return [(q.strip(), a.strip()) for q, a in matches]

def save_to_jsonl(pairs, output_file):
    """
    Função auxiliar para escrever uma lista de pares no arquivo (append).
    """
    with open(output_file, "a", encoding="utf-8") as f:
        for q, a in pairs:
            f.write(json.dumps({"question": q, "answer": a}, ensure_ascii=False) + "\n")

def generate_dataset(
    pdf_path,
    model="gemma4:31b-cloud",
    output_file="dataset.jsonl",
    max_chunks=None,
    chunk_size=1800,
    questions_per_chunk=5,
    max_question_tokens=50,
    max_answer_tokens=50,
    temperature=1.0,
    top_p=0.95,
    repeat_penalty=1.15,
    flush_interval=1   # a cada N chunks, dá flush (1 = após cada chunk)
):
    print("Extracting PDF...")
    text = extract_text_from_pdf(pdf_path)
    chunks = split_text(text, chunk_size)
    if max_chunks:
        chunks = chunks[:max_chunks]

    total_pairs = 0

    # Abre o arquivo uma vez no modo "w" (sobrescreve se existir)
    # Se quiser retomar de um arquivo existente, mude para "a" e ajuste a lógica
    with open(output_file, "w", encoding="utf-8") as f_out:
        for idx, chunk in enumerate(tqdm(chunks, desc="Generating", unit="chunk"), 1):
            try:
                pairs = generate_instruction_response(
                    chunk,
                    model=model,
                    questions_per_chunk=questions_per_chunk,
                    max_question_tokens=max_question_tokens,
                    max_answer_tokens=max_answer_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    repeat_penalty=repeat_penalty
                )
                # Escreve cada par imediatamente
                for q, a in pairs:
                    f_out.write(json.dumps({"question": q, "answer": a}, ensure_ascii=False) + "\n")
                    total_pairs += 1

                # Flush periódico para garantir que os dados cheguem ao disco
                if idx % flush_interval == 0:
                    f_out.flush()

                # Opcional: mostrar progresso a cada 10 chunks
                if idx % 10 == 0:
                    tqdm.write(f"Chunk {idx}/{len(chunks)} - Total pairs saved: {total_pairs}")

            except Exception as e:
                print(f"Error processing chunk {idx}: {e}")
                # continua com o próximo chunk

        # Garante que tudo seja escrito ao final
        f_out.flush()

    print(f"Done! Total {total_pairs} QA pairs saved to {output_file}")

if __name__ == "__main__":
    generate_dataset(
        pdf_path="CAT-4083-UK.pdf",
        model="llama3.2:latest",
        output_file="dataset.jsonl",
        max_chunks=100,
        chunk_size=100,
        questions_per_chunk=1,
        max_question_tokens=50,
        max_answer_tokens=50,
        temperature=0.8,
        top_p=0.95,
        repeat_penalty=1.15,
        flush_interval=5   # flush a cada 5 chunks (ajuste conforme necessidade)
    )